In [1]:
import pandas as pd

In [2]:
df_log = pd.read_csv('data/full_event_log.csv', parse_dates=['timestamp', 'timestamp_end'])

In [13]:
df_case = df_log[df_log['case_id'] == 'CA100']
df_case = df_case.reset_index().rename(columns={'index': 'id'})
df_case

,id,case_id,activity,timestamp,timestamp_end
0,0,CA100,Traslado a carga,2024-02-27 14:01:02,2024-02-27 14:05:56
1,1,CA100,Carga,2024-02-27 14:05:56,2024-02-27 14:07:51
2,2,CA100,Traslado a descarga,2024-02-27 14:07:51,2024-02-27 14:29:21
3,3,CA100,Descarga,2024-02-27 14:29:21,2024-02-27 14:30:14
4,4,CA100,Traslado a carga,2024-02-27 14:30:14,2024-02-27 14:56:13
...,...,...,...,...,...
5049,5049,CA100,Traslado a descarga,2024-03-26 07:50:16,2024-03-26 08:16:23
5050,5050,CA100,Carga de combustible,2024-03-26 08:07:54,2024-03-26 08:16:45
5051,5051,CA100,Traslado a descarga,2024-03-26 08:16:23,2024-03-26 08:16:43
5052,5052,CA100,Descarga,2024-03-26 08:16:43,2024-03-26 08:17:34


In [24]:
df_case_lagged = df_case[['id', 'activity', 'timestamp', 'timestamp_end']].shift(-1).dropna()
df_case_lagged['id'] = df_case['id'].astype(int)
df_case_lagged = df_case_lagged.rename(columns={'activity': 'future_activity', 'timestamp': 'future_timestamp', 'timestamp_end': 'future_timestamp_end'})
df_join = pd.merge(df_case, df_case_lagged, on='id').dropna()

In [52]:
# Actividad que comienza después de que la anterior termine
case_total_events = df_case.shape[0]
df_overlapping_events = df_join[df_join['timestamp_end'] > df_join['future_timestamp']][['activity', 'future_activity']]
print(f'{df_overlapping_events.shape[0] / case_total_events:.2%}')
df_overlapping_events['activity'].value_counts()

21.15%


activity
Traslado a descarga          307
Traslado a carga             272
Pista obstruida              143
Carga                         73
Cambio de turno               52
Sin equipo de carguio         50
Operador fuera del equipo     40
Colacion cabina               36
Descarga                      25
Carga de combustible          22
Colacion comedor              15
Problema operador              9
Mantencion imprevista          7
Otra demora                    6
Otro tipo de reserva           6
Otra mantencion                3
Traslado a colación            2
Mantencion programada          1
Name: count, dtype: int64

- Esto dice que en un ~21% de las actividades en este caso (`case_id` como `truck_id`) la actividad siguiente comienza, mientras no termina la anterior, esto produce paralelismo de las actividades

In [49]:
# See what activities are contained completed
df_contained_events = df_join[(df_join['timestamp'] <= df_join['future_timestamp']) &\
                              (df_join['timestamp_end'] >= df_join['future_timestamp_end'])
                             ]
contained_events = df_contained_events.shape[0]
print(f'{contained_events / case_total_events:.2%}')
df_contained_events['activity'].value_counts()

12.78%


activity
Traslado a descarga          175
Traslado a carga             160
Pista obstruida               91
Cambio de turno               41
Sin equipo de carguio         36
Colacion cabina               35
Operador fuera del equipo     27
Carga                         18
Carga de combustible          16
Colacion comedor              14
Problema operador              9
Otro tipo de reserva           6
Mantencion imprevista          6
Otra demora                    4
Descarga                       3
Otra mantencion                3
Mantencion programada          1
Traslado a colación            1
Name: count, dtype: int64